In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'dsl' / 'executor.py').is_file())
sys.path.append(str(project_root))

OUT = project_root / "plots" / "fig4_1"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_IMAGE = "CLEVR_val_000138"
QUESTION = "There is another object that is the same shape as the brown rubber thing; what is its color?"
DSL_LINEAR = "scene | filter_color brown | filter_material rubber | unique | same_shape | unique | query_color"
ANSWER = "yellow"

clevr_path = project_root / "datasets" / "converted" / "clevr" / "clevr_val.jsonl"
with open(clevr_path) as f:
    for line in f:
        d = json.loads(line)
        if d["image_id"] == TARGET_IMAGE:
            sample = d; break

image_path = project_root / sample["image_path"]
img_np = np.array(Image.open(image_path).convert("RGB"))
print(f"Слика: {TARGET_IMAGE}")


In [ ]:
from scene_graph.builder import SceneGraphBuilder
from scene_graph.relations import RelationConfig

builder = SceneGraphBuilder(relation_config=RelationConfig())
scene_graph, interm = builder.build(
    image_path, use_segmentation=True, use_depth=True,
    use_heuristics=True, return_intermediates=True,
)
detections = interm.get("detections", [])
segments   = interm.get("segments", [])
depth_map  = interm.get("depth_map")
print(f"Детектовано {len(detections)} објеката")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_np)
ax.axis("off")
plt.tight_layout(pad=0)
plt.savefig(OUT / "a_original.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_np)
ax.axis("off")
cmap = plt.get_cmap("tab10")
for i, det in enumerate(detections):
    color = cmap(i % 10)
    x1, y1, x2, y2 = det["bbox"]
    ax.add_patch(mpatches.FancyBboxPatch(
        (x1, y1), x2-x1, y2-y1,
        boxstyle="round,pad=2", linewidth=4,
        edgecolor=color, facecolor=color, alpha=0.15, zorder=3))
    ax.add_patch(mpatches.FancyBboxPatch(
        (x1, y1), x2-x1, y2-y1,
        boxstyle="round,pad=2", linewidth=4,
        edgecolor=color, facecolor="none", alpha=0.95, zorder=4))
    conf = det.get("confidence", 0)
    ax.text(x1+4, y1-4, f"Сигурност: {conf*100:.0f}%",
            ha="left", va="bottom", fontsize=8, color="white", weight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6, lw=0), zorder=8)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    ax.plot(cx, cy, "o", markersize=16, color=color,
            markeredgecolor="white", markeredgewidth=1.5, zorder=6)
    ax.text(cx, cy, str(i), ha="center", va="center",
            fontsize=9, weight="bold", color="white", zorder=7)
plt.tight_layout(pad=0)
plt.savefig(OUT / "b_detection.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fastsam_segs = builder._segmenter.segment(image_path, detections)
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_np, alpha=0.45)
ax.axis("off")
cmap_seg = plt.get_cmap("tab10")
for i, seg in enumerate(fastsam_segs):
    mask = seg.get("mask")
    if mask is None: continue
    color = np.array(cmap_seg(i % 10)[:3])
    overlay = np.zeros((*mask.shape, 4), dtype=float)
    overlay[mask > 0] = [*color, 0.6]
    ax.imshow(overlay, zorder=3)
    ys, xs = np.where(mask > 0)
    if len(xs):
        cx, cy = xs.mean(), ys.mean()
        ax.plot(cx, cy, "o", markersize=14, color=color,
                markeredgecolor="white", markeredgewidth=1.5, zorder=5)
        ax.text(cx, cy, str(i), ha="center", va="center",
                fontsize=8, weight="bold", color="white", zorder=6)
plt.tight_layout(pad=0)
plt.savefig(OUT / "c_masks.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img_np, alpha=0.45)
ax.axis("off")
cmap_seg = plt.get_cmap("tab10")
for i, seg in enumerate(segments):
    mask = seg.get("mask")
    if mask is None: continue
    color = np.array(cmap_seg(i % 10)[:3])
    overlay = np.zeros((*mask.shape, 4), dtype=float)
    overlay[mask > 0] = [*color, 0.6]
    ax.imshow(overlay, zorder=3)
    ys, xs = np.where(mask > 0)
    if len(xs):
        cx, cy = xs.mean(), ys.mean()
        ax.plot(cx, cy, "o", markersize=14, color=color,
                markeredgecolor="white", markeredgewidth=1.5, zorder=5)
        ax.text(cx, cy, str(i), ha="center", va="center",
                fontsize=8, weight="bold", color="white", zorder=6)
plt.tight_layout(pad=0)
plt.savefig(OUT / "c2_zbuffer.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
if depth_map is not None:
    dm = depth_map.astype(float)
    dm_norm = (dm - dm.min()) / (dm.max() - dm.min() + 1e-8)
    ax.imshow(dm_norm, cmap="plasma")
ax.axis("off")
plt.tight_layout(pad=0)
plt.savefig(OUT / "d_depth.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
from scene_graph.renderer import render_scene_graph
fig = render_scene_graph(image_path, scene_graph, title="", show_image=False)
plt.savefig(OUT / "e_scene_graph.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
steps = sample["program"]
n = len(steps)
fig_h = n * 0.55 + 0.5
fig, ax = plt.subplots(figsize=(5.5, fig_h))
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.axis("off")
ax.add_patch(mpatches.FancyBboxPatch(
    (0.03, 0.02), 0.94, 0.96,
    boxstyle="round,pad=0.02", linewidth=1.5,
    edgecolor="#cccccc", facecolor="#f8f8f8",
    transform=ax.transAxes, zorder=1))
for i, step in enumerate(steps):
    fn = step["function"]
    vi = step.get("value_inputs", [])
    y = 1 - (i + 0.8) / n
    ax.text(0.10, y, f"[{i}]", transform=ax.transAxes,
            fontsize=14, family="monospace", color="#aaaaaa", va="center")
    ax.text(0.24, y, fn, transform=ax.transAxes,
            fontsize=14, family="monospace", color="#1a1a2e",
            weight="bold", va="center")
    if vi:
        offset = 0.062 * len(fn)
        ax.text(0.24 + offset, y, f"  →  {vi[0]}", transform=ax.transAxes,
                fontsize=14, family="monospace", color="#c05621", va="center")
plt.tight_layout(pad=0)
plt.savefig(OUT / "f_dsl_steps.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import cv2

target_det = next(s for s in segments if s.get("mask") is not None)
x1,y1,x2,y2 = [int(v) for v in target_det["bbox"]]
img_bgr = cv2.imread(str(image_path))
crop = img_bgr[y1:y2, x1:x2].copy()

seg_model = builder._segmenter
results = seg_model.model(
    source=crop, device=seg_model.device,
    retina_masks=True, imgsz=seg_model.imgsz,
    conf=seg_model.conf, iou=seg_model.iou, verbose=False,
)
masks_raw = results[0].masks.data.cpu().numpy()
areas = masks_raw.sum(axis=(1,2))
raw_mask = masks_raw[int(np.argmax(areas))] > 0.5

m = (raw_mask.astype(np.uint8) * 255)
kernel = np.ones((3, 3), np.uint8)
m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  kernel, iterations=2)
m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel, iterations=1)
morph_mask = m > 0

crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mask, label in zip(axes, [raw_mask, morph_mask],
                            ["Sirova maska (FastSAM)", "Nakon morfoloskih operacija"]):
    ax.imshow(crop_rgb, alpha=0.5)
    overlay = np.zeros((*mask.shape, 4), dtype=float)
    overlay[mask > 0] = [0.2, 0.6, 1.0, 0.65]
    ax.imshow(overlay, zorder=3)
    ax.axis("off")
plt.tight_layout(pad=0)
plt.savefig(OUT / "c1_morph_compare.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.patheffects as pe
objs = scene_graph["objects"]
rels = scene_graph["relationships"]
n = len(objs)
OBJECT_COLORS = {
    "red":"#FF4444","blue":"#4499FF","green":"#44CC44","yellow":"#FFCC00",
    "cyan":"#00CCDD","purple":"#AA44CC","brown":"#AA6633","gray":"#999999",
}
xs = [o["pixel_coords"][0] for o in objs]
ds = [o["pixel_coords"][2] if len(o["pixel_coords"]) > 2 else 0 for o in objs]
xr = max(xs)-min(xs) or 1; dr = max(ds)-min(ds) or 1
xs_n = [(x-min(xs))/xr for x in xs]
ds_n = [(d-min(ds))/dr for d in ds]
fig, ax = plt.subplots(figsize=(8, 7))
ax.set_facecolor("white")
ax.set_xlim(-0.18, 1.18); ax.set_ylim(-0.22, 1.18)
ax.grid(True, alpha=0.3, color="#cccccc", linestyle="--")
REL_STYLE = {"left": ("#4499FF","←"), "right": ("#FF4444","→"),
             "front": ("#44CC44","↓"), "behind": ("#FFAA00","↑")}
drawn = set()
for rel_name, (rel_color, rel_sym) in REL_STYLE.items():
    for i, targets in enumerate(rels.get(rel_name, [])):
        for j in targets:
            if (i, j, rel_name) in drawn: continue
            drawn.add((i, j, rel_name))
            ax.annotate("", xy=(xs_n[j], ds_n[j]), xytext=(xs_n[i], ds_n[i]),
                arrowprops=dict(arrowstyle="->", color=rel_color, lw=1.2, alpha=0.5))
cmap = plt.get_cmap("tab10")
for i, (obj, xn, dn) in enumerate(zip(objs, xs_n, ds_n)):
    color = OBJECT_COLORS.get(obj.get("color"), "#aaaaaa")
    ax.scatter(xn, dn, s=1100, color="white", zorder=3)
    ax.scatter(xn, dn, s=900, color=color, zorder=4)
    ax.text(xn, dn, str(i), ha="center", va="center",
            fontsize=13, weight="bold", color="white", zorder=5)
    lbl = f"{obj.get('color','?')} {obj.get('shape','?')}"
    ax.annotate(lbl, xy=(xn, dn), xytext=(0, -28), textcoords="offset points",
        ha="center", va="top", fontsize=9, color="#222222",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#eeeeee", alpha=0.9, lw=0), zorder=6)
from matplotlib.lines import Line2D
legend_els = [Line2D([0],[0], color=col, lw=2, label=f"{sym} {nm}")
              for nm,(col,sym) in REL_STYLE.items()]
ax.legend(handles=legend_els, fontsize=8, loc="upper left")
ax.set_xticks([0,0.5,1]); ax.set_xticklabels(["лево","центар","десно"], fontsize=8)
ax.set_yticks([0,0.5,1]); ax.set_yticklabels(["иза","средина","испред"], fontsize=8)
ax.set_xlabel("← лево          десно →", fontsize=9, color="#333333")
ax.set_ylabel("← иза        испред →", fontsize=9, color="#333333")
plt.tight_layout(pad=0)
plt.savefig(OUT / "h_relations.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
from dsl.executor import Executor
from dsl.scene_graph import SceneGraph

sg = SceneGraph(sample["scene_graph"])
result = Executor(sg).execute(sample["program"])

print("=" * 60)
print("ПИТАЊЕ:")
print(f"  {QUESTION}")
print()
print("DSL ПРОГРАМ (линеаризован):")
print(f"  {DSL_LINEAR}")
print()
print("DSL ПРОГРАМ (кораци):")
for i, step in enumerate(sample["program"]):
    vi = step.get("value_inputs", [])
    vi_str = f"  [{vi[0]}]" if vi else ""
    print(f"  [{i}] {step['function']}{vi_str}  <- {step['inputs']}")
print()
print(f"ОДГОВОР: {result}")
print("=" * 60)
print(f"Све слике сачуване у: {OUT}")
